# `parse_question` — Brique 2 : parser une question utilisateur

Référence : chapitre **« Understanding the Question Before Searching »** + design Kezhan dans `docs/06_question_layer.md`.

## Sortie

**Toujours une `list[dict]`** (1 question simple = 1 entrée). Chaque entrée :

| Section | Contenu |
|---|---|
| `retrieval`  | `main_query`, `page_hint`, `section_hint`, `layout_hint`, `document_hint`, `anchor_keywords` |
| `generation` | `original_question`, `format_constraint`, `disambiguation`, `must_distinguish` |
| `_meta`      | `intent`, `document_type`, `bricks_active` (traçabilité) |

**Le JSON ne contient QUE des champs populés** (pas de `null`).

## Connaissance métier dans les PRESETS

| `document_type` | Briques actives |
|---|---|
| `pdf`   | toutes |
| `word`  | sans `page_hint` (pas de pagination stable en .docx) |
| `excel` | sans `page_hint` ni `section_hint` (axes orthogonaux : feuilles + colonnes) |
| `email` | sans `page_hint` ni `section_hint` |
| `pptx`  | toutes (slide = page) |

Override fin par appel : `parse_question(q, enable={"page_hint": False})`.

## Approche

Regex / heuristique uniquement (pas de LLM, conforme à `CLAUDE.md`). Compatible drop-in avec la sortie de `src.question.understand_question` (même structure `[{retrieval, generation, _meta}]`).

In [ ]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ../../../..

In [ ]:
import json
from docpipeline.question_parsing import parse_question

QUESTIONS = [
    "Quelle est la date d'effet du contrat ? Page 1, format YYYY-MM-DD.",
    "What is the limit per claim, not the deductible, in the Limits section page 5? Article L131-1 applies.",
    "Look in the exclusions section for the flooding clause.",
    "Compare the indemnification and liability caps in the latest version.",
    "List all the obligations of the seller, in bullet list.",
    "Find the policy number — it's usually in the header, format JSON.",
    "Dans le tableau page 3 du contrat MAIF, quel est le montant ? En euros.",
    "Combien coûte l'assurance ?",
]

for q in QUESTIONS:
    plan = parse_question(q, document_type='pdf')
    print(f'Q: {q}')
    print(json.dumps(plan, indent=2, ensure_ascii=False))
    print('-' * 80)